In [1]:
from transformers import pipeline

model = "Qwen/Qwen3.5-2B"

generator = pipeline(
    "text-generation",
     model=model,
     max_length=None,
     max_new_tokens=150, 
     do_sample=True,
     return_full_text=False
)

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_length', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [1]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root))
from app.app import load_resources

2026-04-16 13:52:32.838 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 13:52:32.838 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-16 13:52:32.839 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-16 13:52:32.839 WARNING streamlit.runtime.caching.cache_data_api: No runtime found, using MemoryCacheStorageManager
2026-04-16 13:52:32.840 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 13:52:32.867 
  command:

    streamlit run /Users/randalllee/miniforge3/envs/dsci-575-project/lib/python3.12/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-04-16 13:52:32.867 Thread 'MainThread': missing ScriptRunContext! This w

In [3]:
documents, bm25 = load_resources()

2026-04-16 13:48:31.762 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 13:48:31.763 No runtime found, using MemoryCacheStorageManager


In [4]:
from src.semantic import create_faiss_index

vectorstore = create_faiss_index(documents, 10000, reload_index=False)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [11]:
from langchain_huggingface import HuggingFacePipeline

llm = HuggingFacePipeline(pipeline=generator)

In [17]:
SYSTEM_PROMPT = """
    /no_think
    You are a helpful Amazon shopping assistant.
    Answer the question using ONLY the following context (real product reviews + metadata).
    Always cite the product ASIN when possible."""

def build_prompt(query, context):
    return f"""{SYSTEM_PROMPT}

context:
{context}

question: 
{query}

Recommend ONE product using the context.
Do not add additional explanations or repeat the prompt.
Stop after the recommendation.

Return the answer exactly in this format:

Product Title:
Product ASIN:
Product Rating:
Product Review:
Reason for Recommendation: Write 2 natural sentences describing the product’s key benefits using evidence from the review and rating.

END
"""

In [18]:
def build_context(docs):
    return "\n\n".join(
        f"Product ASIN: {doc.metadata.get('asin')}\n"
        f"Product Title: {doc.metadata.get('product_title')}\n"
        f"Product Rating: {doc.metadata.get('product_rating')}\n"
        f"Product Review: {doc.metadata.get('product_review')}\n"
        for doc in docs
    )

In [19]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

format_context = RunnableLambda(build_context)
def prompt_builder(inputs):
    return build_prompt(inputs["input"], inputs["context"])

prompt = RunnableLambda(prompt_builder)


rag_chain = (
    {
        "context": retriever | format_context,
        "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
query = "Moisturizing shampoo for thick curly hair"

response = rag_chain.invoke(query)

print(response)

</think>

Product Title: Verb Ghost Shampoo & Conditioner Duo
Product ASIN: B097BSKSQW
Product Rating: 5.0
Product Review: I have curly, but fine hair. Although it looks thick, it isn't! So many products for curly hair assume that you have loads of coarse, thick hair, but that's not how it works! This product is moisturizing but not heavy. It is perfect! This is now my holy grail shampoo/condish combo! POW POW for fine curly girls!
Reason for Recommendation: This moisturizing and anti-frizz duo is perfect for fine curly hair that needs hydration without being heavy. It is praised as a "holy grail" because


In [2]:
from src.rag_pipeline import RAGPipeline

rag = RAGPipeline()

response = rag.ask("something to keep your face moisturized all day")

print(response)

2026-04-16 13:52:42.569 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-16 13:52:42.570 No runtime found, using MemoryCacheStorageManager


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/320 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_length', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.






    <think>
Thinking Process:

1.  **Analyze the Request:**
    *   Role: Helpful Amazon shopping assistant.
    *   Task: Recommend ONE product based on the provided context.
    *   Input Context: Three product listings (ASIN, Title, Rating, Review).
    *   Question: "something to keep your face moisturized all day"
    *   Output Format: Specific template (Product Title, Product ASIN, Product Rating, Product Review, Reason for Recommendation).
    *   Constraint: Use ONLY the provided context.
    *   Constraint: Always cite the product ASIN when possible.
    *   Constraint: Write 2 natural sentences describing the key benefits
